# 01. Unión del modelo canónico

Unir los siete archivos sin modificar sus registros ni eliminar duplicados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

entrada = "../02_modelo_canonico/02_segunda_version"
salida = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"

archivos = {
    "Canonico_1_V2.csv": 245,
    "Canonico_2_V2.csv": 122,
    "Canonico_3_V2.csv": 710,
    "Canonico_4_V2.csv": 178,
    "Canonico_5_V2.csv": 580,
    "Canonico_6_V2.csv": 318,
    "Canonico_ORCID_V2.csv": 429
}

columnas = ["Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
            "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi", "URL",
            "Area", "SubArea", "Keywords", "Abstract"]

## 1. Lectura y validación

In [ ]:
datos = {}

for archivo, filas in archivos.items():
    df = pd.read_csv(f"{entrada}/{archivo}", dtype=str, keep_default_na=False)
    assert len(df) == filas, f"Cantidad de filas incorrecta: {archivo}"
    assert list(df.columns) == columnas, f"Columnas incorrectas: {archivo}"
    datos[archivo] = df

union = pd.concat(datos.values(), ignore_index=True)
assert union.shape == (2582, 15)

# Comprobar que cada bloque conserva los datos de su archivo original.
inicio = 0
for archivo, df in datos.items():
    fin = inicio + len(df)
    assert union.iloc[inicio:fin].reset_index(drop=True).equals(df.reset_index(drop=True)), archivo
    inicio = fin

union.head(3)

## 2. Registros por archivo

In [ ]:
resumen = pd.DataFrame({"Registros": {archivo: len(df) for archivo, df in datos.items()}})
resumen.index.name = "Archivo"
resumen["Porcentaje"] = (100 * resumen["Registros"] / len(union)).round(2)
resumen

In [ ]:
ax = resumen["Registros"].plot.barh(figsize=(9, 4))
ax.invert_yaxis()
ax.bar_label(ax.containers[0], padding=4)
ax.set_xlim(0, resumen["Registros"].max() * 1.15)
ax.set_title("Registros aportados por cada archivo")
ax.set_xlabel("Número de registros")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Guardado y comprobación

In [ ]:
union.to_csv(salida, index=False, encoding="utf-8-sig")
verificacion = pd.read_csv(salida, dtype=str, keep_default_na=False)
assert verificacion.equals(union), "El archivo guardado difiere de la unión."

print(f"Total de registros: {len(verificacion):,}")
print(f"Columnas: {verificacion.shape[1]}")
print("Integridad de los datos: OK")
print(f"Archivo generado: {salida}")